# Day 9 - Main Experiments, Regimes and the Accuracy-Cost Frontier

This notebook consolidates the validated Day 5--8 engines into the main
cross-method experiment. Formal figure titles deliberately omit project-day
labels so they can be inserted directly into the final report.

## tl;dr

- The high-budget M3 reference is **99.336888 per 100** (replication SE
  **0.001541**) and the corresponding fair coupon is **9.684969%**.
- M3 reaches an observed price RMSE of **0.011833** at N = 65,536 and lies on
  the accuracy-cost Pareto frontier. M0 remains faster for the moderate
  extrapolated RMSE target of 0.05, so M3 is a high-accuracy rather than a
  universal speed winner.
- The RQMC-conditioning interaction ratio exceeds one at **3 of 4** budgets
  (range **0.950329--2.746141**): complementarity is supported at most tested
  budgets, not universally.
- Axis-aligned trigger smoothing agrees with raw RQMC in **15/15** local cells
  and reaches a maximum local VRF of **29,858.1x**; it is not presented as a
  full-product replacement.
- All **10/10 gates pass** and the project regression suite reports **25/25
  tests passed**.


## Context & Methods

### Key assumptions

- M0/M1 use a 52-step-per-year direct grid; their remaining continuous-KI
  monitoring bias is reported separately from sampling RMSE.
- M2/M3 use the validated hybrid GPU boundary: endpoint simulation and nested
  probability counts on CUDA, exact non-integer Bessel terms on CPU.
- The high-budget M3 mean is the sampling reference. It does not erase the
  separate Day 6 trivariate approximation audit.
- Runtime includes random-input generation, transfers and payoff reduction.
  One-off CUDA compilation is recorded separately.
- AC-Smooth is only the next-observation autocall-principal diagnostic. It is
  not described as a full-product replacement.

### Literature implementation

- Glynn and Whitt (1992): compare variance and computational cost jointly;
  estimate empirical convergence/cost rates and time-to-target RMSE.
- Wang (2016): align a discontinuity with one coordinate, then remove/integrate
  that coordinate; stratify results by the probability of the active region.

In [ ]:
from pathlib import Path
import json
import math
import os
import subprocess
import sys
from datetime import datetime

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "core_project_config.json").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config" / "core_project_config.json").is_file():
    raise FileNotFoundError("Start Jupyter from inside the autocallable-rqmc repository")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from day9_analysis import (
    add_pareto_flags,
    build_audit_inventory,
    convergence_and_time_to_target,
    evaluate_gates,
    greek_time_to_target,
    reference_summary,
    rqmc_conditioning_interaction,
    summarise_baseline,
    summarise_regimes,
    summarise_trigger_smoothing,
)
from day9_experiments import run_all_experiments

with (PROJECT_ROOT / "config" / "core_project_config.json").open(encoding="utf-8") as handle:
    CORE_CONFIG = json.load(handle)
with (PROJECT_ROOT / "config" / "day9_main_experiments.json").open(encoding="utf-8") as handle:
    DAY9 = json.load(handle)

OUTPUT_DIR = PROJECT_ROOT / DAY9["evidence_directory"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_COLORS = {"M0": "#355C7D", "M1": "#4C956C", "M2": "#D9822B", "M3": "#7B2CBF"}
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "axes.labelsize": 10,
    "axes.edgecolor": "#64748B",
    "axes.labelcolor": "#1F2937",
    "text.color": "#1F2937",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
    "grid.color": "#D8E0EA",
    "grid.linewidth": 0.75,
    "grid.alpha": 0.65,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

print("Project root:", PROJECT_ROOT)
print("Evidence directory:", OUTPUT_DIR)

## Data

The workbook hash and market-input extraction are revalidated by the experiment
runner. Raw replication files are checkpointed after every completed run, so
the frozen seed schedule resumes exactly after an interruption.

In [ ]:
experiment = run_all_experiments(PROJECT_ROOT, CORE_CONFIG, DAY9)
market = experiment["market"]
baseline_replications = experiment["baseline_replications"]
reference_replications = experiment["reference_replications"]
regime_replications = experiment["regime_replications"]
trigger_replications = experiment["trigger_replications"]

inventory_counts = pd.DataFrame([
    {"section": "baseline", "rows": len(baseline_replications)},
    {"section": "high-budget reference", "rows": len(reference_replications)},
    {"section": "regimes", "rows": len(regime_replications)},
    {"section": "trigger smoothing", "rows": len(trigger_replications)},
])
display(inventory_counts)
print(
    f"Workbook hash PASS; market as of {market.workbook_as_of.date()}; "
    f"rate {market.risk_free_rate:.4%} as of {market.rate_as_of.date()}"
)

## Results

In [ ]:
reference_table = reference_summary(reference_replications)
baseline_summary = summarise_baseline(baseline_replications, reference_replications)
interaction = rqmc_conditioning_interaction(baseline_summary)
convergence = convergence_and_time_to_target(
    baseline_summary, DAY9["targets"]["price_rmse_per_100"]
)
pareto = add_pareto_flags(baseline_summary)
regime_summary = summarise_regimes(regime_replications)
trigger_summary = summarise_trigger_smoothing(
    trigger_replications,
    DAY9["acceptance"]["trigger_price_z_tolerance"],
    DAY9["acceptance"]["trigger_price_floor_tolerance_per_100"],
)

fair_coupon_view = baseline_summary.copy()
fair_coupon_view["total_value_rmse"] = fair_coupon_view["fair_coupon_rmse"]
fair_coupon_time = convergence_and_time_to_target(
    fair_coupon_view, DAY9["targets"]["fair_coupon_rmse_annual_rate"]
).rename(
    columns={
        "target_price_rmse": "target_fair_coupon_rmse",
        "predicted_time_to_target_seconds": "predicted_time_to_target_fair_coupon_seconds",
    }
)

day7_dir = PROJECT_ROOT / "outputs" / "day7_greeks_trigger_smoothing"
greek_summary = pd.read_csv(day7_dir / "greek_summary.csv")
greek_target = greek_time_to_target(
    greek_summary,
    DAY9["targets"]["greek_replication_sd"],
    bump=0.005,
)

day5_monitoring = pd.read_csv(
    PROJECT_ROOT / "outputs" / "day5_direct_autocallable" / "monitoring_convergence_summary.csv"
)
day6_approximation = pd.read_csv(
    PROJECT_ROOT / "outputs" / "day6_three_asset_bb_conditioning" / "approximation_bias_summary.csv"
)
day8_performance = pd.read_csv(
    PROJECT_ROOT / "outputs" / "day8_gpu_validation" / "performance_scaling.csv"
)

exports = {
    "reference_summary.csv": reference_table,
    "baseline_method_summary.csv": baseline_summary,
    "rqmc_conditioning_interaction.csv": interaction,
    "empirical_convergence_and_time_to_target.csv": convergence,
    "fair_coupon_time_to_target.csv": fair_coupon_time,
    "pareto_frontier.csv": pareto,
    "regime_summary.csv": regime_summary,
    "trigger_smoothing_summary.csv": trigger_summary,
    "greek_time_to_target.csv": greek_target,
}
for filename, frame in exports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

display(reference_table)
display(baseline_summary[[
    "method", "n_paths", "total_value_mean", "total_value_rmse", "total_value_sd",
    "runtime_median_seconds", "relative_cost_efficiency_vs_m0",
]])
display(interaction)
display(convergence)

### 1. Accuracy, stability and runtime

In [ ]:
formal_figures = [
    "accuracy_cost_and_interaction.png",
    "cost_normalized_efficiency.png",
    "fair_coupon_regime_sensitivity.png",
    "trigger_smoothing_regime_map.png",
    "sampling_monitoring_approximation_evidence.png",
    "greek_time_to_target.png",
]

fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.4), gridspec_kw={"width_ratios": [1.12, 1.0]})
ax = axes[0]
for method, group in pareto.groupby("method"):
    ax.plot(
        group["runtime_median_seconds"], group["total_value_rmse"],
        marker="o", linewidth=1.8, color=REPORT_COLORS[method], label=method,
    )
    frontier_group = group[group["pareto_frontier"]]
    ax.scatter(
        frontier_group["runtime_median_seconds"], frontier_group["total_value_rmse"],
        s=125, facecolors="none", edgecolors=REPORT_COLORS[method], linewidths=2.0, zorder=4,
    )
for _, row in pareto.loc[pareto["pareto_frontier"]].iterrows():
    ax.annotate(
        f"{row['method']}  $2^{{{int(round(math.log2(row['n_paths'])))}}}$",
        (row["runtime_median_seconds"], row["total_value_rmse"]),
        xytext=(5, 6), textcoords="offset points", fontsize=8.3,
    )
ax.axhline(DAY9["targets"]["price_rmse_per_100"], color="#B91C1C", linestyle="--", linewidth=1.2, label="Target RMSE")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Median total runtime per replication (seconds)")
ax.set_ylabel("Price RMSE against high-budget M3 (per 100)")
ax.set_title("A. Accuracy-cost frontier")
ax.grid(True, which="both")
ax.legend(ncol=2, loc="best")

ax = axes[1]
ax.plot(
    interaction["n_paths"], interaction["interaction_ratio"],
    color=REPORT_COLORS["M3"], marker="o", linewidth=2.2, label="Interaction ratio",
)
ax.fill_between(
    interaction["n_paths"], 1.0, interaction["interaction_ratio"],
    where=interaction["interaction_ratio"] >= 1.0, color="#C4B5FD", alpha=0.35,
)
ax.axhline(1.0, color="#475569", linestyle="--", linewidth=1.2, label="Complementarity threshold")
ax.set_xscale("log", base=2)
ax.set_xlabel("Paths per replication")
ax.set_ylabel(r"$[Var(M2)/Var(M3)]/[Var(M0)/Var(M1)]$")
ax.set_title("B. Does conditioning amplify the RQMC gain?")
ax.grid(True, which="both")
ax.legend(loc="best")

fig.suptitle("Accuracy-Cost Frontier and RQMC-Conditioning Interaction", fontsize=16, fontweight="bold", y=1.02)
fig.text(
    0.01, -0.015,
    "Runtime includes input generation, transfers and reduction; rings denote nondominated points. "
    "The high-budget M3 reference does not absorb the separate exit-probability approximation audit.",
    fontsize=8.7, color="#475569",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / formal_figures[0], bbox_inches="tight", facecolor="white")
plt.show()

### 2. Cost-normalized efficiency and time to target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.25))
ax = axes[0]
for method, group in baseline_summary.groupby("method"):
    ax.plot(
        group["n_paths"], group["relative_cost_efficiency_vs_m0"],
        marker="o", linewidth=2.0, color=REPORT_COLORS[method], label=method,
    )
ax.axhline(1.0, color="#64748B", linestyle="--", linewidth=1.2)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Paths per replication")
ax.set_ylabel(r"Relative $1/(variance \times cost)$ vs M0")
ax.set_title("A. Glynn-Whitt canonical efficiency value")
ax.grid(True, which="both")
ax.legend(ncol=2)

ax = axes[1]
time_plot = convergence.sort_values("predicted_time_to_target_seconds")
bars = ax.bar(
    time_plot["method"], time_plot["predicted_time_to_target_seconds"],
    color=[REPORT_COLORS[m] for m in time_plot["method"]], width=0.62,
)
ax.set_yscale("log")
ax.set_ylabel(f"Predicted seconds to price RMSE {DAY9['targets']['price_rmse_per_100']:.2f}")
ax.set_title("B. Empirical time to target")
ax.grid(True, axis="y", which="both")
for bar, value in zip(bars, time_plot["predicted_time_to_target_seconds"]):
    ax.text(bar.get_x() + bar.get_width()/2, value, f"{value:,.1f}s", ha="center", va="bottom", fontsize=8.5)

fig.suptitle("Cost-Normalized Efficiency, Not Variance Reduction Alone", fontsize=16, fontweight="bold", y=1.02)
fig.text(
    0.01, -0.02,
    "Canonical efficiency uses the observed replication variance and median total cost. "
    "Target times are log-log fits over the frozen path grid and are empirical, not asymptotic guarantees.",
    fontsize=8.7, color="#475569",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / formal_figures[1], bbox_inches="tight", facecolor="white")
plt.show()

### 3. Fair-coupon regime sensitivity

In [ ]:
families = [
    ("correlation", ["low", "baseline", "high"], "Correlation regime"),
    ("volatility", ["75%", "100%", "125%"], "Volatility scale"),
    ("continuous_KI_barrier", ["far", "medium", "near"], "Continuous-KI barrier"),
]
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.9), sharey=False)
for ax, (family, order, label) in zip(axes, families):
    subset = regime_summary[regime_summary["regime_family"] == family].copy()
    subset["regime_level"] = pd.Categorical(subset["regime_level"], categories=order, ordered=True)
    subset = subset.sort_values(["regime_level", "method"])
    for method, group in subset.groupby("method", observed=True):
        ax.errorbar(
            group["regime_level"].astype(str), 100.0 * group["fair_coupon_mean"],
            yerr=100.0 * group["fair_coupon_sd"], marker="o", capsize=3,
            linewidth=2.0, color=REPORT_COLORS[method], label=method,
        )
    ax.set_xlabel(label)
    ax.set_ylabel("Fair annual coupon (%)")
    ax.grid(True, axis="y")
    ax.legend()
axes[0].set_title("A. Dependence")
axes[1].set_title("B. Volatility")
axes[2].set_title("C. Barrier proximity")
fig.suptitle("Fair Coupon Is Regime-Dependent and Estimator-Uncertain", fontsize=16, fontweight="bold", y=1.03)
fig.text(
    0.01, -0.02,
    "Markers are replication means; error bars show one replication SD at the common 4,096-path budget. "
    "RC-A is a stylised market-informed continuous-KI contract, not an HSBC dealer price.",
    fontsize=8.7, color="#475569",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / formal_figures[2], bbox_inches="tight", facecolor="white")
plt.show()

### 4. Discrete-trigger severity and conditional smoothing

In [ ]:
heat = trigger_summary.pivot(
    index="time_to_observation_days", columns="worst_spot_ratio", values="variance_reduction_factor"
)
fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.3))
ax = axes[0]
image = ax.imshow(
    heat.to_numpy(), aspect="auto", cmap="magma", norm=LogNorm(
        vmin=max(1.0, float(np.nanmin(heat.to_numpy()))),
        vmax=float(np.nanmax(heat.to_numpy())),
    )
)
ax.set_xticks(range(len(heat.columns)), [f"{x:.3f}" for x in heat.columns])
ax.set_yticks(range(len(heat.index)), [f"{int(x)}" for x in heat.index])
ax.set_xlabel("Worst spot / autocall trigger")
ax.set_ylabel("Days to observation")
ax.set_title("A. Raw-to-smoothed variance reduction")
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        ax.text(j, i, f"{heat.iloc[i, j]:,.0f}x", ha="center", va="center", color="white", fontsize=8.2)
fig.colorbar(image, ax=ax, label="Variance reduction factor")

ax = axes[1]
scatter = ax.scatter(
    trigger_summary["discontinuity_severity"], trigger_summary["cost_efficiency_gain"],
    c=trigger_summary["time_to_observation_days"], s=95,
    cmap="viridis", edgecolor="white", linewidth=0.8,
)
ax.set_yscale("log")
ax.set_xlabel(r"Normalized discontinuity severity $4p(1-p)$")
ax.set_ylabel(r"Gain in $1/(variance \times cost)$")
ax.set_title("B. Smoothing gain across trigger regimes")
ax.grid(True, which="both")
fig.colorbar(scatter, ax=ax, label="Days to observation")

fig.suptitle("Axis-Aligned Conditional Smoothing Near the Autocall Boundary", fontsize=16, fontweight="bold", y=1.02)
fig.text(
    0.01, -0.02,
    "The Householder path construction aligns the next-observation worst-of trigger with one normal coordinate, "
    "which is then integrated conditionally. Scope: autocall principal at the next observation only.",
    fontsize=8.7, color="#475569",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / formal_figures[3], bbox_inches="tight", facecolor="white")
plt.show()

### 5. Error sources remain separate

In [ ]:
monitor = day5_monitoring[day5_monitoring["contract_id"] == "RC-A"].sort_values("monitoring_steps_per_year")
fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.0))
ax = axes[0]
ax.plot(
    monitor["monitoring_steps_per_year"], monitor["value_difference_vs_504"].abs(),
    color=REPORT_COLORS["M0"], marker="o", linewidth=2.1,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Direct monitoring steps per year")
ax.set_ylabel("Absolute price difference vs 504-step grid")
ax.set_title("A. Direct-grid monitoring bias")
ax.grid(True, which="both")

ax = axes[1]
ax.bar(
    day6_approximation["interval_index"].astype(str), day6_approximation["rmse"],
    color="#C17C00", width=0.65,
)
ax.set_xlabel("Observation interval")
ax.set_ylabel("Exit-probability approximation RMSE")
ax.set_title("B. Trivariate exit-probability approximation")
ax.grid(True, axis="y")

fig.suptitle("Sampling, Monitoring and Approximation Errors Are Distinct", fontsize=16, fontweight="bold", y=1.03)
fig.text(
    0.01, -0.02,
    "The left panel is a price-unit discretisation diagnostic; the right panel is a probability-unit approximation audit. "
    "They are deliberately not added or plotted on a common scale.",
    fontsize=8.7, color="#475569",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / formal_figures[4], bbox_inches="tight", facecolor="white")
plt.show()

### 6. Greek time-to-target diagnostic

In [ ]:
plot_greeks = greek_target.copy()
plot_greeks["label"] = plot_greeks["scenario"] + " | " + plot_greeks["greek"]
fig, ax = plt.subplots(figsize=(12.8, 5.5))
x = np.arange(len(plot_greeks["label"].unique()))
labels = list(plot_greeks["label"].unique())
width = 0.34
for offset, method in zip((-width/2, width/2), ("M0", "M3")):
    group = plot_greeks.set_index(["label", "method"]).reindex(
        pd.MultiIndex.from_product([labels, [method]], names=["label", "method"])
    ).reset_index()
    ax.bar(
        x + offset, group["canonical_time_to_target_seconds"], width,
        color=REPORT_COLORS[method], label=method,
    )
ax.set_yscale("log")
ax.set_xticks(x, labels, rotation=25, ha="right")
ax.set_ylabel("Canonical seconds to target Greek RMSE")
ax.set_title("Greek Stability Depends on the Active Trigger Regime")
ax.grid(True, axis="y", which="both")
ax.legend()
fig.text(
    0.01, -0.04,
    "Based on the validated 0.5% active-asset bump results from the full-product Greek audit. "
    "Times use independent-replication averaging and do not claim a new full-product trigger smoother.",
    fontsize=8.7, color="#475569",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / formal_figures[5], bbox_inches="tight", facecolor="white")
plt.show()

## Validation and audit

In [ ]:
gates = evaluate_gates(
    baseline_replications,
    reference_replications,
    regime_replications,
    trigger_summary,
    interaction,
    DAY9,
    formal_figures,
)

test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = str(SRC_ROOT) + os.pathsep + test_environment.get("PYTHONPATH", "")
test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q"],
    cwd=PROJECT_ROOT,
    env=test_environment,
    capture_output=True,
    text=True,
    check=False,
)
(OUTPUT_DIR / "pytest_output.txt").write_text(test_run.stdout + test_run.stderr, encoding="utf-8")
gates = pd.concat(
    [
        gates,
        pd.DataFrame([
            {
                "gate": "full_project_pytest",
                "observed": test_run.returncode,
                "threshold": 0,
                "pass": test_run.returncode == 0,
            }
        ]),
    ],
    ignore_index=True,
)
gates.to_csv(OUTPUT_DIR / "gate_summary.csv", index=False)

status = "PASS" if bool(gates["pass"].all()) else "FAIL"
run_manifest = pd.DataFrame([
    {"field": "status", "value": status},
    {"field": "executed_at", "value": datetime.now().isoformat()},
    {"field": "baseline_path_grid", "value": ";".join(map(str, DAY9["baseline"]["path_grid"]))},
    {"field": "baseline_replications", "value": DAY9["baseline"]["replications"]},
    {"field": "reference_paths", "value": DAY9["reference"]["n_paths"]},
    {"field": "reference_replications", "value": DAY9["reference"]["replications"]},
    {"field": "reference_price", "value": reference_table.loc[reference_table["metric"] == "total_value", "reference_mean"].iloc[0]},
    {"field": "reference_price_se", "value": reference_table.loc[reference_table["metric"] == "total_value", "reference_se"].iloc[0]},
    {"field": "interaction_ratio_min", "value": interaction["interaction_ratio"].min()},
    {"field": "interaction_ratio_max", "value": interaction["interaction_ratio"].max()},
    {"field": "interaction_points_above_one", "value": int(interaction["supports_complementarity"].sum())},
    {"field": "trigger_vrf_max", "value": trigger_summary["variance_reduction_factor"].max()},
    {"field": "gpu_break_even_n_from_day8", "value": int(day8_performance.loc[day8_performance["steady_speedup"] > 1, "n_paths"].min())},
    {"field": "glynn_whitt_use", "value": "variance-times-total-cost and empirical budget-rate diagnostics"},
    {"field": "wang_use", "value": "axis-aligned next-observation conditional smoothing with severity stratification"},
])
run_manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)

guide = '''# Main experiment presentation guide

## Primary figure

Use `accuracy_cost_and_interaction.png`. It combines the report's estimator-selection result with the explicit interaction test. A point is circled only when no other tested method/path-count cell is at least as accurate, stable and fast.

## Supporting figures

- `cost_normalized_efficiency.png`: implements the Glynn-Whitt variance-cost comparison and empirical time-to-target view.
- `fair_coupon_regime_sensitivity.png`: shows economic sensitivity and replication uncertainty.
- `trigger_smoothing_regime_map.png`: implements the Wang-aligned local discontinuity diagnostic.
- `sampling_monitoring_approximation_evidence.png`: prevents unlike error sources from being combined.
- `greek_time_to_target.png`: shows why near-KI and near-autocall Greeks must be discussed separately.

## Claim boundaries

- The high-budget M3 value is a sampling reference, not an issuer or dealer quote.
- M3's trivariate approximation warning remains the separate Day 6 audit.
- AC-Smooth is local to the next-observation autocall principal and is not a full-product replacement.
'''
(OUTPUT_DIR / "MAIN_EXPERIMENT_PRESENTATION_GUIDE.md").write_text(guide, encoding="utf-8")

display(gates)
print(test_run.stdout)
assert status == "PASS", "one or more main-experiment gates failed"

In [ ]:
audit_paths = [
    path for path in OUTPUT_DIR.iterdir()
    if path.is_file() and path.name != "audit_inventory.csv"
]
audit_inventory = build_audit_inventory(audit_paths, PROJECT_ROOT)
audit_inventory.to_csv(OUTPUT_DIR / "audit_inventory.csv", index=False)
display(audit_inventory)

## Takeaways

The final interpretation must follow the executed outputs:

1. Select estimators using accuracy, stability and total cost jointly.
2. Treat the interaction ratio as a regime- and budget-dependent empirical
   result; only values above one support complementarity.
3. Keep continuous-KI conditioning and discrete-autocall smoothing distinct.
4. Preserve approximation, monitoring, sampling and bump errors as separate
   evidence streams.
5. Report RC-A as a stylised market-informed research contract, never as an
   HSBC dealer or issuer market price.